# Lab 05-02 · NumPy, Vectorization and Numerical Computing

Companion to chapter [05-02](../../chapters/05-02-numpy-and-vectorization.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. Part 1's timings depend on your computer; everything else is deterministic (random numbers use fixed seeds).

In [1]:
import math
import time
import warnings
import numpy as np
print(np.__version__)

2.5.4


## Part 1 — Loop versus array

**🔮 Predict:** how many times faster is NumPy than a Python loop at computing the RMS of a million samples?

In [2]:
rng = np.random.default_rng(2026)
samples = rng.normal(0.0, 1.0, 1_000_000)
as_list = samples.tolist()

t0 = time.perf_counter()
acc = 0.0
for s in as_list:
    acc += s * s
rms_loop = math.sqrt(acc / len(as_list))
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
rms_np = float(np.sqrt(np.mean(samples ** 2)))
t_np = time.perf_counter() - t0
print(f"loop {t_loop * 1e3:.1f} ms, numpy {t_np * 1e3:.2f} ms, speedup about {t_loop / t_np:.0f}x")
print(f"RMS: loop {rms_loop:.6f}, numpy {rms_np:.6f}, agree {math.isclose(rms_loop, rms_np, rel_tol=1e-12)}")

loop 48.5 ms, numpy 4.18 ms, speedup about 12x
RMS: loop 1.000125, numpy 1.000125, agree True


Expect a speedup of roughly 10× to 100×, depending on the machine and the operation (squaring and averaging a million numbers is near the low end, because NumPy makes a temporary array for the squares). Both give the same RMS, close to 1 because the samples were drawn with standard deviation 1.

## Part 2 — Broadcasting a design space

**🔮 Predict:** how many E12 resistor (1 kΩ decade) and capacitor (100 nF decade) pairs give a time constant within 2% of 1 ms?

In [3]:
e12 = np.array([1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2])
r = e12[:, None] * 1e3          # ohms
c = e12[None, :] * 100e-9       # farads
tau = r * c                     # 12 x 12 table of time constants
ok = np.abs(tau - 1e-3) / 1e-3 < 0.02
for i, j in zip(*np.nonzero(ok)):
    print(f"R = {r[i, 0]:5.0f} ohm, C = {c[0, j] * 1e9:4.0f} nF, tau = {tau[i, j] * 1e3:.3f} ms")
print("pairs:", int(ok.sum()))

R =  1200 ohm, C =  820 nF, tau = 0.984 ms
R =  1500 ohm, C =  680 nF, tau = 1.020 ms
R =  1800 ohm, C =  560 nF, tau = 1.008 ms
R =  5600 ohm, C =  180 nF, tau = 1.008 ms
R =  6800 ohm, C =  150 nF, tau = 1.020 ms
R =  8200 ohm, C =  120 nF, tau = 0.984 ms
pairs: 6


Six pairs qualify, in three mirror-image couples (1.2 × 8.2, 1.5 × 6.8 and 1.8 × 5.6, each in both orders). One broadcast expression replaced a double loop over 144 combinations.

## Part 3 — Number formats

**🔮 Predict:** how many decimal digits does float16 carry, and what is its largest value?

In [4]:
for t in (np.float16, np.float32, np.float64):
    fi = np.finfo(t)
    print(f"{t.__name__:8} eps {float(fi.eps):.3e}  max {float(fi.max):.4e}  ~{fi.precision} decimal digits")
for t in (np.int8, np.uint8, np.int16, np.int32, np.int64):
    ii = np.iinfo(t)
    print(f"{t.__name__:6} {ii.min} ... {ii.max}")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(np.float16(65504) * np.float16(2), np.float16(1) + np.float16(0.0004))

float16  eps 9.766e-04  max 6.5504e+04  ~3 decimal digits
float32  eps 1.192e-07  max 3.4028e+38  ~6 decimal digits
float64  eps 2.220e-16  max 1.7977e+308  ~15 decimal digits
int8   -128 ... 127
uint8  0 ... 255
int16  -32768 ... 32767
int32  -2147483648 ... 2147483647
int64  -9223372036854775808 ... 9223372036854775807
inf 1.0


float16 has about 3 decimal digits and overflows above 65504: doubling 65504 gives `inf`, and adding 0.0004 to 1 changes nothing because the gap above 1.0 is about 0.001. This is why AI training in 16-bit formats needs care (scaling losses, accumulating in 32 bits) and why bfloat16 keeps float32's range at the cost of precision.

## Part 4 — NaN and summation

**🔮 Predict:** what is the mean of `[1, 2, nan]`? And which of a loop, `np.sum` and `math.fsum` gets closest to the true sum of a million float32 values of 0.1?

In [5]:
x = np.array([1.0, 2.0, np.nan])
print(x.mean(), np.nanmean(x), np.nan == np.nan, np.isnan(x))
a = np.full(1_000_000, 0.1, dtype=np.float32)
loop = np.float32(0.0)
for v in a:
    loop += v
exact = math.fsum(a.astype(np.float64))
print(f"loop {float(loop):.2f}  np.sum {float(a.sum()):.2f}  sum in float64 {a.sum(dtype=np.float64):.5f}  exact {exact:.5f}")

nan 1.5 False [False False  True]


loop 100958.34  np.sum 100000.01  sum in float64 100000.00149  exact 100000.00149


The float32 loop drifts almost 1% high; `np.sum`'s pairwise algorithm stays within 0.01; accumulating in float64 matches the exact sum of the stored values.

## Part 5 — Reproducible randomness

**🔮 Predict:** will two generators created with the same seed give the same numbers? What about two created without a seed?

In [6]:
a1 = np.random.default_rng(42).normal(size=3)
a2 = np.random.default_rng(42).normal(size=3)
b1 = np.random.default_rng().normal(size=3)
b2 = np.random.default_rng().normal(size=3)
print(np.round(a1, 6), np.array_equal(a1, a2), np.array_equal(b1, b2))

[ 0.304717 -1.039984  0.750451] True False


Seeded generators reproduce exactly; unseeded ones draw a fresh seed from the operating system each time. Seed every simulation whose result you will report or test, and vary the seed deliberately to check that a conclusion does not depend on it.

## Your turn

1. In Part 2, widen the tolerance to 5% and change the target to 4.7 ms. How many pairs qualify?
2. In Part 4, replace 0.1 with 0.5 (exact in binary). Does the float32 loop still drift? Why or why not?

<details><summary>Answers</summary>

1. Three pairs: 5.6 kΩ with 820 nF (4.592 ms), 6.8 kΩ with 680 nF (4.624 ms) and 8.2 kΩ with 560 nF (4.592 ms). The obvious 4.7 kΩ with 1 µF is not in the grid, because the capacitor array covers only the 100 nF decade (100 to 820 nF); widen the arrays to two decades (`np.concatenate([e12, 10 * e12])`) to include it. Change one line of the broadcast expression and the whole table is re-evaluated: that is the point of working with arrays.
2. No: the loop gives exactly 500000.0, as does `np.sum`. 0.5 is exact in binary, and every running total is a multiple of 0.5 below $2^{19}$, which needs at most 20 significant bits; float32 has 24. Push it further, though, and float32 fails differently: once the total reaches $2^{23}$ = 8,388,608, the spacing between neighbouring float32 values is 1, so adding 0.5 is an exact tie that rounds back to the even total, and the sum stops growing. Summing 40 million halves in float32 returns 8388608.0 instead of 20,000,000. Exact inputs do not make an accumulator safe; its width must cover the total.

</details>